# Ponte Câmbio: integridade do relatório

Revisão do preliminar. O e-mail é fonte documental da Ponte, não substitui as decisões atualizadas do financeiro. Duplicatas são preservadas; a simulação sem repetições mede impacto, sem autorizar exclusão. Não se vinculam PIX a clientes por prefixos de nome.

In [1]:
from pathlib import Path
from decimal import Decimal
import csv, io, re, hashlib, unicodedata
from collections import Counter
from datetime import datetime
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs/FORMATO_SAIDA.md').exists())
PASTA = RAIZ / 'Tarvo_Data_Pack_set2026'  # altere para outro pacote; nomes são descobertos abaixo
assert PASTA.is_dir(), f'Pacote ausente: {PASTA}'
def arquivo(padrao):
    encontrados = list(PASTA.glob(padrao))
    assert len(encontrados) == 1, (padrao, encontrados)
    return encontrados[0]
def texto(path):
    bruto = path.read_bytes()
    try: return bruto.decode('utf-8-sig'), 'utf-8-sig'
    except UnicodeDecodeError: return bruto.decode('cp1252'), 'cp1252'
def dinheiro(s):
    s = str(s).strip()
    assert re.fullmatch(r'-?(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}', s), s
    return Decimal(s.replace('.', '').replace(',', '.'))
def total(valores): return sum(valores, Decimal('0.00'))
def norm(s): return ''.join(c for c in unicodedata.normalize('NFKD', s.upper()) if not unicodedata.combining(c))
pd.set_option('display.max_colwidth', 100)


In [2]:
import email, email.policy
msg = email.message_from_bytes(arquivo('03_*.eml').read_bytes(), policy=email.policy.default)
corpo = msg.get_body(preferencelist=('plain',)).get_content()
print(corpo)
anexos = [p for p in msg.iter_attachments() if (p.get_filename() or '').lower().endswith('.xlsx')]
assert len(anexos) == 1
xls = pd.ExcelFile(io.BytesIO(anexos[0].get_payload(decode=True)))
print(xls.sheet_names)
bruto = pd.read_excel(xls, 'Operações', header=4)
ops = bruto[bruto['Nº Operação'].astype(str).str.startswith('PC-')].copy()
ops['linha_planilha'] = ops.index + 6
display(ops.head())

Prezados,

Segue em anexo o relatório mensal de operações de câmbio do parceiro TARVO PAGAMENTOS referente à competência 09/2026.

Conforme contrato de parceria, o repasse da participação de 50% (cinquenta por cento) do spread realizado será creditado em 06/10/2026, mediante recebimento da nota fiscal de intermediação.

Lembramos que operações com status "APROVADO C/ RESSALVA" seguem normalmente para liquidação.

Em caso de divergência, favor responder este e-mail em até 5 dias úteis.

Atenciosamente,
Backoffice Parceiros - Ponte Câmbio S.A. - CNPJ 75.374.179/0001-46
Este é um e-mail automático.

['Operações', 'Resumo']


,Nº Operação,Data Contratação,Data Liquidação,CNPJ Cliente,Cliente,Moeda,Valor ME,Taxa Cliente,PTAX Ref.,Valor R$,Spread R$,Spread %,Spread Estimado R$ (D-1),IOF R$,Status Compliance,Observação,linha_planilha
0,PC-0926-0002,2026-09-30 00:00:00,2026-10-01,34671260000100,MAPLE STUDY ASSESSOR,NZD,5402.22,3.2796,3.2311,17717.12,262.01,0.015010,279.09,67.33,APROVADO,NaN,6
1,PC-0926-0003,2026-09-10 00:00:00,2026-09-11,14.760.206/0001-24,NÓRDICA EDUCAÇÃO INT,USD,7929.38,5.5281,5.4599,43834.41,540.78,0.012491,455.08,166.57,APROVADO,NaN,7
2,PC-0926-0004,2026-09-02 00:00:00,2026-09-03,61.720.397/0001-80,LISBOA CURSOS E VIAG,USD,5733.03,5.4114,5.3262,31023.72,488.45,0.015996,480.85,117.89,APROVADO,NaN,8
3,PC-0926-0007,2026-09-29 00:00:00,2026-09-30,14.760.206/0001-24,NÓRDICA EDUCAÇÃO INT,CAD,5289.90,4.0096,3.9601,21210.38,261.85,0.012500,346.59,80.60,APROVADO,NaN,9
4,PC-0926-0011,2026-09-04 00:00:00,2026-09-07,17217888000111,MUNDO ABERTO INTERCÂ,CAD,2438.46,4.0920,4.0315,9978.18,147.53,0.015007,99.27,37.92,APROVADO,NaN,10


## Duplicatas e valores

Nº Operação repetido deve ser revisado com os campos completos e a observação. Valores lidos do Excel são convertidos pela representação decimal e somados sem aritmética binária adicional.

In [3]:
for campo in ['Valor R$','Spread R$','IOF R$']:
    ops[campo] = ops[campo].map(lambda v: Decimal(str(v)))
dup = ops[ops['Nº Operação'].duplicated(keep=False)]
display(dup)
print('Linhas:', len(ops), '| operações distintas:', ops['Nº Operação'].nunique())
cenarios = {'Todas as linhas':ops, 'Simulação: primeira ocorrência':ops.drop_duplicates('Nº Operação')}
display(pd.DataFrame([{'cenario':nome,'linhas':len(base),'spread':total(base['Spread R$']),'volume_brl':total(base['Valor R$'])} for nome,base in cenarios.items()]))
display(ops[ops['Spread R$'].map(lambda v:v < 0)])
datas_c = pd.to_datetime(ops['Data Contratação'])
datas_l = pd.to_datetime(ops['Data Liquidação'])
display(ops[datas_c.dt.to_period('M') != datas_l.dt.to_period('M')])

,Nº Operação,Data Contratação,Data Liquidação,CNPJ Cliente,Cliente,Moeda,Valor ME,Taxa Cliente,PTAX Ref.,Valor R$,Spread R$,Spread %,Spread Estimado R$ (D-1),IOF R$,Status Compliance,Observação,linha_planilha
70,PC-0926-0162,2026-09-22 00:00:00,2026-09-23,50900987000179,BAGAGEM LIVRE TURISM,USD,10429.99,5.3308,5.252,55600.19,821.88,0.015004,931.75,211.28,APROVADO,NaN,76
71,PC-0926-0162,2026-09-22 00:00:00,2026-09-23,50900987000179,BAGAGEM LIVRE TURISM,USD,10429.99,5.3308,5.252,55600.19,821.88,0.015004,931.75,211.28,APROVADO,reprocessada,77


Linhas: 233 | operações distintas: 232


,cenario,linhas,spread,volume_brl
0,Todas as linhas,233,78073.59,5645804.52
1,Simulação: primeira ocorrência,232,77251.71,5590204.33


,Nº Operação,Data Contratação,Data Liquidação,CNPJ Cliente,Cliente,Moeda,Valor ME,Taxa Cliente,PTAX Ref.,Valor R$,Spread R$,Spread %,Spread Estimado R$ (D-1),IOF R$,Status Compliance,Observação,linha_planilha
90,PC-0926-0210,2026-09-15 00:00:00,2026-09-16,95739378000118,PONTE AÉREA ESTUDANT,EUR,7087.81,6.1518,6.2488,43602.79,-687.52,-0.015523,587.97,165.69,APROVADO,Taxa travada no pedido (D-1),96
135,PC-0926-0326,2026-09-03 00:00:00,2026-09-04,53.857.560/0001-23,CANGURU INTERCÂMBIOS,CAD,8213.30,3.9321,3.9941,32295.52,-509.22,-0.015523,473.84,122.72,APROVADO,Taxa travada no pedido (D-1),141
199,PC-0926-0491,2026-09-23 00:00:00,2026-09-24,68567965000102,DUBLIN HUB INTERCÂMB,NZD,5904.65,3.1415,3.1910,18549.46,-292.28,-0.015512,331.48,70.49,APROVADO,Taxa travada no pedido (D-1),205
222,PC-0926-0545,2026-09-14 00:00:00,2026-09-15,66564512000199,NOVO HORIZONTE INTER,AUD,5918.24,3.4765,3.5348,20574.76,-345.03,-0.016493,229.61,78.18,APROVADO,Taxa travada no pedido (D-1),228


,Nº Operação,Data Contratação,Data Liquidação,CNPJ Cliente,Cliente,Moeda,Valor ME,Taxa Cliente,PTAX Ref.,Valor R$,Spread R$,Spread %,Spread Estimado R$ (D-1),IOF R$,Status Compliance,Observação,linha_planilha
0,PC-0926-0002,2026-09-30 00:00:00,2026-10-01,34671260000100,MAPLE STUDY ASSESSOR,NZD,5402.22,3.2796,3.2311,17717.12,262.01,0.015010,279.09,67.33,APROVADO,NaN,6
49,PC-0926-0110,2026-09-30 00:00:00,2026-10-01,34671260000100,MAPLE STUDY ASSESSOR,EUR,4555.70,6.2348,6.1427,28403.88,419.58,0.014993,528.44,107.93,APROVADO,NaN,55
105,PC-0926-0249,2026-09-30 00:00:00,2026-10-01,95927868000148,ROTA GLOBAL INTERCÂM,CAD,5491.97,3.9377,3.8795,21625.73,319.63,0.015002,301.84,82.18,APROVADO,NaN,111
159,PC-0926-0384,2026-09-30 00:00:00,2026-10-01,95.739.378/0001-18,PONTE AÉREA ESTUDANT,USD,5064.02,5.5408,5.4589,28058.72,414.74,0.015003,394.03,106.62,APROVADO,NaN,165
223,PC-0926-0549,2026-09-30 00:00:00,2026-10-01,95927868000148,ROTA GLOBAL INTERCÂM,AUD,5285.50,3.5747,3.5219,18894.08,279.07,0.014992,352.09,71.8,APROVADO,NaN,229
230,PC-0926-0566,2026-09-30 00:00:00,2026-10-01,17217888000111,MUNDO ABERTO INTERCÂ,AUD,7118.34,3.6470,3.5931,25960.59,383.68,0.015001,274.13,98.65,APROVADO,NaN,236


## Totais declarados e consequências para a saída

O rodapé e a aba Resumo precisam ser comparados com as linhas, mantendo os dois cenários. Spread negativo e contratação em competência diferente da liquidação exigem decisão documentada. Não assumir spread padrão pela mediana cobrada, nem excluir repetição apenas porque melhora a conciliação.

In [4]:
display(bruto[~bruto['Nº Operação'].astype(str).str.startswith('PC-')].dropna(how='all'))
display(pd.read_excel(xls,'Resumo'))
print('Spread total:', total(ops['Spread R$']))
print('Spread negativo:', total(ops.loc[ops['Spread R$'].map(lambda x:x<0),'Spread R$']))

,Nº Operação,Data Contratação,Data Liquidação,CNPJ Cliente,Cliente,Moeda,Valor ME,Taxa Cliente,PTAX Ref.,Valor R$,Spread R$,Spread %,Spread Estimado R$ (D-1),IOF R$,Status Compliance,Observação
234,Total de operações:,233,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235,Spread total R$:,78073.59,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Moeda,Qtd,Volume ME,Spread R$
0,USD,37,176345.36,14007.38
1,EUR,33,187134.41,16074.48
2,GBP,36,169456.27,17739.02
3,CAD,42,202407.79,10705.21
4,AUD,48,234433.14,11329.45
5,NZD,37,185680.20,8218.05


Spread total: 78073.59
Spread negativo: -1834.05


`comissoes_por_lojista.csv` precisa preservar Nº Operação e a decisão sobre duplicatas e negativos em seu rastro. `notas_a_emitir.csv` depende da base e do tomador aprovados. Comissão recebida no banco não comprova receita do mês. Não gerar provisões nesta exploração.

Pendências de cadastro, taxa e contrato cabem ao comercial; classificação e liberação, ao financeiro, sujeitos ao documento atualizado ainda ausente. O módulo futuro deve ler anexos em memória, manter linha da planilha e Nº Operação e separar base bruta de cenários de revisão.